# Jupiter to automize clustering for Cats

In [2]:
import pandas as pd
import numpy as np
import sklearn.metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import autosklearn.classification

## Overture des données

In [3]:
df = pd.read_csv('data/OutCatdata.csv', na_filter= False)
df = df.drop("Unnamed: 0", axis= 1)

/tmp/ipykernel_95801/3855532020.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/OutCatdata.csv', na_filter= False)


/!\\ the NA values are dropped /!\\

## observation des données

In [3]:
df

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,Hunt,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


### division des jeux de données entre la classe à prédire (ici `Hunt`) et les autres variables

In [4]:
variables = df.drop(['Hunt'], axis = 1)
variables

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


In [5]:
classes = df['Hunt']
classes

0          Yes
1          Yes
2          Yes
3          Yes
4          Yes
          ... 
1057315    Yes
1057316    Yes
1057317    Yes
1057318    Yes
1057319    Yes
Name: Hunt, Length: 1057320, dtype: object

comme prévu, nous avons 2 tableaux : 

* classes : soit les classes à prédire (`Hunt`)
* variables : soit les variables du tableau

## Création du modèle qualitatif

In [6]:
cls = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=60, 
    per_run_time_limit=120, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

*les param minimum pour la tâche* : 

- time_left_for_this_task= 2000 s
- per_run_time_limit=30 cycles
- n_jobs = 16 coeur
- memory_limit = 24 Go

### Génération des jeux de test, validation

pour notre entrainement, nous prenons des proportions de 67% de test et 33% de test

Les méthodes testées sont : 

* Forêt aléatoire
* Latent Dirichlet Allocation
* Multilayered Perceptron
* Baisien naif
* k plus proches voisins 

### Dans un premier temps, nous allons utiliser seulement les données Quantitatives

#### gestion de la suppression des colonnes quantitatives

In [7]:
dfQuali = variables.drop(["event.id","timestamp","location.long","location.lat",
                          "animal.id","StartDate","StartHours","EndDate","EndHours"],
                         axis = 1).astype('category')

Crée le jeu de test qualitatif

In [8]:
variables_trainQ, variables_testQ, classes_trainQ, classes_testQ = train_test_split(
        																dfQuali, classes, test_size = 0.5, random_state=0)

Transformation de classesQ en `category` à la place de `object`

In [9]:
classes_testQ = classes_testQ.astype("category")
classes_trainQ = classes_trainQ.astype("category")

application des paramêtre afin de crée le modèle

In [10]:
cls.fit(variables_trainQ, classes_trainQ)

/usr/local/lib/python3.8/dist-packages/autosklearn/data/target_validator.py:187: UserWarning: Fitting transformer with a pandas series which has the dtype category. Inverse transform may not be able preserve dtype when converting to np.ndarray
  warnings.warn(


[WARNING] [2025-04-27 06:51:58,010:Client-AutoML(1):1a89c39e-2334-11f0-a0fd-121c485a7426] Time limit for a single run is higher than total time limit. Capping the limit for a single run to the total time given to SMAC (59.371650)
[WARNING] [2025-04-27 06:51:58,010:Client-AutoML(1):1a89c39e-2334-11f0-a0fd-121c485a7426] Capping the per_run_time_limit to 29.0 to have time for a least 2 models in each process.


Process pynisher function call:
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/sklearn/utils/_encode.py", line 132, in _unique_python
    uniques = sorted(uniques_set)
TypeError: '<' not supported between instances of 'str' and 'int'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.8/dist-packages/pynisher/limit_function_call.py", line 133, in subprocess_func
    return_value = ((func(*args, **kwargs), 0))
  File "/usr/local/lib/python3.8/dist-packages/autosklearn/smbo.py", line 160, in _calculate_metafeatures_encoded
    result = calculate_all_metafeatures_encoded_labels(
  File "/usr/local/lib/python3.8/dist-packages/autosklearn/metalearning/metafeatures

[WARNING] [2025-04-27 06:51:59,097:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 412 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 102 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 367 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 262 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 37 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 605 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 88 not found
[WARNING] [2025-04-27 06:51:59,098:Client-AutoMLSMBO(1)::1a89c39e-2334-11f0-a0fd-121c485a7426] Configuration 426 not found
[WARNING] [2025-04

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=120,
                      time_left_for_this_task=60)

Ne fournis pas de résultats pertinant, potentielement par manque de temps de calcul (25 et 30min avait été testé)

In [11]:
cls.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
1,1,1.0,<NA>,<NA>,<NA>


Ici, nous voyons que le modèle n'est même pas arriver à faire une ligne (facteur séparateur)

In [12]:
predictions_Hunt = list(cls.predict(variables_testQ))

précision : 

In [13]:
print("Accuracy score:", sklearn.metrics.accuracy_score(np.array(classes_testQ), predictions_Hunt))

Accuracy score: 0.14220292815798433


table des stats

In [14]:
print( sklearn.metrics.classification_report(classes_testQ, predictions_Hunt) )

/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

          NA       0.14      1.00      0.25     75177
          No       0.00      0.00      0.00     75219
         Yes       0.00      0.00      0.00    378264

    accuracy                           0.14    528660
   macro avg       0.05      0.33      0.08    528660
weighted avg       0.02      0.14      0.04    528660



/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


décevant : on a un précision catastrophique.

Nous allons regarder la matrice de confusion pour potentiellement observer quel groupe est le mieu prédit

In [15]:
np.round( confusion_matrix(classes_testQ, predictions_Hunt), 3)

array([[ 75177,      0,      0],
       [ 75219,      0,      0],
       [378264,      0,      0]])

Le problème est clair : on prédit tout en une classe

## Tentative avec du quantitatif

### Ouverture du csv

In [4]:
dfQuanti = pd.read_csv('data/OutCatdataQuantiNormZ.csv', na_filter= False)
dfQuanti = dfQuanti.drop("Unnamed: 0", axis= 1)

/tmp/ipykernel_95801/2793842605.py:1: DtypeWarning: Columns (6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  dfQuanti = pd.read_csv('data/OutCatdataQuantiNormZ.csv', na_filter= False)


### Suppression des lignes ayant des Na

In [5]:
dfQuanti = dfQuanti[~np.any(dfQuanti == "NA",axis=1)]

### Création des sous jeux de données de test et d'entrainement

In [6]:
x = dfQuanti.drop('Hunt', axis=1).to_numpy().astype(np.float64)
y = dfQuanti.Hunt.astype(object)

y[y ==	-2.16472428387967] = "Na"
y[y == 	-0.78922734140023] = "No"
y[y ==   0.586269601079213]  = "Yes"

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size = 0.5, random_state=0)

### Établissement d'un modèle

In [19]:
cls_hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=900, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

Ajustment du modèle a notre jeu de données

In [20]:
cls_hunt.fit(x_train, y_train, dataset_name='Cat Data')

[WARNING] [2025-04-27 06:53:03,487:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-27 06:53:03,487:Client-AutoMLSMBO(1)::Cat Data] Configuration 367 not found
[WARNING] [2025-04-27 06:53:03,487:Client-AutoMLSMBO(1)::Cat Data] Configuration 426 not found
[WARNING] [2025-04-27 06:53:03,487:Client-AutoMLSMBO(1)::Cat Data] Configuration 88 not found
[WARNING] [2025-04-27 06:53:03,487:Client-AutoMLSMBO(1)::Cat Data] Configuration 605 not found
[WARNING] [2025-04-27 06:53:03,487:Client-AutoMLSMBO(1)::Cat Data] Configuration 37 not found
[WARNING] [2025-04-27 06:53:03,488:Client-AutoMLSMBO(1)::Cat Data] Configuration 69 not found
[WARNING] [2025-04-27 06:53:03,488:Client-AutoMLSMBO(1)::Cat Data] Configuration 206 not found
[WARNING] [2025-04-27 06:53:03,488:Client-AutoMLSMBO(1)::Cat Data] Configuration 262 not found
[WARNING] [2025-04-27 06:53:03,488:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 06:53:03,488:Client-AutoMLSMBO(

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=100,
                      time_left_for_this_task=900)

### Affichage des résultats

### Affichage des facteurs

In [21]:
cls_hunt.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
5,1,0.02,k_nearest_neighbors,0.000000,21.166709
56,4,0.02,k_nearest_neighbors,0.000000,43.436522
64,5,0.04,k_nearest_neighbors,0.000000,21.074136
68,7,0.02,k_nearest_neighbors,0.000000,36.099317
120,10,0.02,k_nearest_neighbors,0.000000,32.381468
129,9,0.04,k_nearest_neighbors,0.000000,41.220996
141,8,0.02,k_nearest_neighbors,0.000000,44.679728
144,6,0.02,k_nearest_neighbors,0.000000,17.541120
155,3,0.06,k_nearest_neighbors,0.000000,24.936812


### Stockage des prédictions

In [22]:
predictions_Hunt = list(cls_hunt.predict(x_test))

### Affichage des stats

In [23]:
print( sklearn.metrics.classification_report(y_test, predictions_Hunt) )

              precision    recall  f1-score   support

          Na       1.00      1.00      1.00     72285
          No       1.00      1.00      1.00     75535
         Yes       1.00      1.00      1.00    377823

    accuracy                           1.00    525643
   macro avg       1.00      1.00      1.00    525643
weighted avg       1.00      1.00      1.00    525643



### Affichage de la matrice de confusion

In [24]:
np.round( confusion_matrix(y_test, predictions_Hunt), 3)

array([[ 72285,      0,      0],
       [     0,  75535,      0],
       [     0,      0, 377823]])

# Prédiction du nombre de proie par chats

## création des 2 matrices

In [7]:
x2 = dfQuanti.drop('N.pray', axis=1).to_numpy().astype(np.float64)
y2 = dfQuanti['N.pray'].astype(object)

### To-do ya des strings dans dfQuanti['N.pray'] 

on les supprimes en copiant dans une liste temp que les float

In [8]:
out = []
for i in y2 : 
    if type(i) == float:
        out.append(i)
    elif type(i) == str :
        out.append(float(i))

y2 = pd.Series(out).astype(object)

### transformer les nombre de poids normalisé en leur valeur d'origine

génération d'un dico de traduction

In [9]:
dictUnfoctor = {}

fooKeys = np.unique(y2)
fooVals = np.unique(df['N.pray'].astype(object))

In [10]:
for i in range(len(fooKeys)):
    dictUnfoctor[fooKeys[i]] = fooVals[i]

application du dico

In [11]:
for pos in range(len(y2)) : 
    y2[pos] = str(dictUnfoctor[y2[pos]])

## On a fini le formatage, c'est l'heure de faire le jeu de test et le jeu d'entrainement

In [12]:
x2_train, x2_test, y2_train, y2_test = train_test_split(x2, y2,
                                                    test_size = 0.5, random_state=0)

## Configuration du modèle

In [31]:
cls_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=800, 
    per_run_time_limit=160, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

### Affinage du modèle

In [32]:
cls_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 434 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 17 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 07:10:29,158:Client-AutoMLSMBO(1)::Cat Data] Configuration 277 not found
[WARNING] [2025-04-27 07:10:29,159:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-27 07:10:29,159:Client-AutoMLSMBO(1)::Cat Data] Configuration 595 not found
[WARNING] [2025-04-27 07:10:29,159:Client-AutoMLSMB

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=160,
                      time_left_for_this_task=800)

## Affichage des résultats

### Affichage des facteurs

In [33]:
cls_Npray.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
6,1,0.08,k_nearest_neighbors,0.000000,19.668080
17,10,0.04,k_nearest_neighbors,0.000000,16.055765
50,11,0.04,k_nearest_neighbors,0.000000,30.312841
52,7,0.04,k_nearest_neighbors,0.000000,29.125421
66,8,0.02,k_nearest_neighbors,0.000000,23.377948
70,6,0.02,k_nearest_neighbors,0.000000,16.620502
82,5,0.02,k_nearest_neighbors,0.000000,18.944720
86,4,0.04,k_nearest_neighbors,0.000000,33.369854
88,3,0.04,k_nearest_neighbors,0.000000,24.788978


On voit que la méthode avec les modeles les plus significatf sont : Ida et gaussian_nb

### Stockage des prédictions

In [34]:
predictions_Npray = list(cls_Npray.predict(x2_test))

### Affichage des stats

In [35]:
print( sklearn.metrics.classification_report(y2_test, predictions_Npray) )

              precision    recall  f1-score   support

          -1       1.00      1.00      1.00    163262
           0       1.00      1.00      1.00    106831
           1       1.00      1.00      1.00     53779
          10       1.00      1.00      1.00     16599
          11       1.00      1.00      1.00      2699
          13       1.00      1.00      1.00      1214
          15       1.00      1.00      1.00      3493
          16       1.00      1.00      1.00       808
          17       1.00      1.00      1.00       561
          19       1.00      1.00      1.00      7307
           2       1.00      1.00      1.00     53685
           3       1.00      1.00      1.00     28708
           4       1.00      1.00      1.00     41843
           5       1.00      1.00      1.00     19360
           6       1.00      1.00      1.00     11099
           7       1.00      1.00      1.00      3248
           8       1.00      1.00      1.00      8978
           9       1.00    

1 de précision pour chaque classes

### Affichage de la matrice de confusion

In [36]:
np.round( confusion_matrix(y2_test, predictions_Npray), 3)

array([[163262,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0, 106831,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,  53779,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,  16599,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,      0,   2699,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,      0,      0,   1214,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
           

problême trop fort : on a une matrice de test diagonalle : aucune mauvaise prédiction, 100% de vrai

donc possibilité de réduire le temps de calcul pour l'entrainement

# Dernière prédiction le temps passé à l'intérieur

## création des 2 matrices

In [13]:
x3 = dfQuanti.drop('Hrs.indors', axis=1).to_numpy().astype(np.float64)
y3 = dfQuanti['Hrs.indors'].astype(object)

on les supprimes en copiant dans une liste temp que les float

In [14]:
out = []
for i in y3 : 
    if type(i) == float:
        out.append((i))
    elif type(i) == str :
        out.append(float(i))

y3 = pd.Series(out).astype(object)

### transformer les nombre de poids normalisé en leur valeur d'origine

génération d'un dico de traduction

In [15]:
dictUnfoctor = {}

fooKeys = np.unique(y3)
fooVals = (df['Hrs.indors'].astype(object))

restriction des données sous forme de string

In [16]:
out = []
for i in fooVals : 
    if type(i) == float:
        out.append(str(i))
    elif type(i) != float :
        out.append((i))

fooVals = pd.unique(out).astype(object)

création dico de traduction

In [17]:
for i in range(len(fooKeys)):
    dictUnfoctor[fooKeys[i]] = fooVals[i]

application du dico

In [18]:
for pos in range(len(y3)) : 
    y3[pos] = str(dictUnfoctor[y3[pos]])

## On a fini le formatage, c'est l'heure de faire le jeu de test et le jeu d'entrainement

In [19]:
x3_train, x3_test, y3_train, y3_test = train_test_split(x3, y3,
                                                    test_size = 0.5, random_state=0)

## Configuration du modèle

In [44]:
cls_HrsOut = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=600, 
    per_run_time_limit=160, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

### Affinage du modèle

In [45]:
cls_HrsOut.fit(x3_train, y3_train, dataset_name='Cat Data')

[WARNING] [2025-04-27 07:26:42,439:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 07:26:42,439:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 07:26:42,439:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 07:26:42,439:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not found
[WARNING] [2025-04-27 07:26:42,439:Client-AutoMLSMBO(1)::Cat Data] Configuration 17 not found
[WARNING] [2025-04-27 07:26:42,439:Client-AutoMLSMBO(1)::Cat Data] Configuration 434 not found
[WARNING] [2025-04-27 07:26:42,440:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 07:26:42,440:Client-AutoMLSMBO(1)::Cat Data] Configuration 124 not found
[WARNING] [2025-04-27 07:26:42,440:Client-AutoMLSMBO(1)::Cat Data] Configuration 494 not found
[WARNING] [2025-04-27 07:26:42,440:Client-AutoMLSMBO(1)::Cat Data] Configuration 277 not found
[WARNING] [2025-04-27 07:26:42,440:Client-AutoMLSMB

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=160,
                      time_left_for_this_task=600)

## Affichage des résultats

### Affichage des facteurs

In [46]:
cls_HrsOut.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
13,1,0.10,k_nearest_neighbors,0.000000,14.986565
68,2,0.02,k_nearest_neighbors,0.000000,12.584594
5,3,0.06,k_nearest_neighbors,0.000006,21.449710
55,8,0.04,k_nearest_neighbors,0.000012,22.666966
57,6,0.08,k_nearest_neighbors,0.000012,64.290606
58,4,0.02,k_nearest_neighbors,0.000012,23.218338
74,5,0.02,k_nearest_neighbors,0.000012,13.712613
77,7,0.04,k_nearest_neighbors,0.000012,18.076830
59,9,0.02,k_nearest_neighbors,0.000190,51.273522


méthode donnant le plus de poids : k_nearest_neighbors

### Stockage des prédictions

In [47]:
predictions_HrsOut = list(cls_HrsOut.predict(x3_test))

plutot "long" car on test +500000 lignes

### Affichage des stats

In [48]:
print( sklearn.metrics.classification_report(y3_test, predictions_HrsOut) )

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6519
           1       1.00      1.00      1.00     65575
          10       1.00      1.00      1.00     41745
          12       1.00      1.00      1.00     43796
          14       1.00      1.00      1.00     13201
          15       1.00      1.00      1.00     36938
          16       1.00      1.00      1.00     16434
          18       1.00      1.00      1.00     53462
           2       1.00      1.00      1.00     52736
          20       1.00      1.00      1.00     12525
          23       1.00      1.00      1.00      7306
           3       1.00      1.00      1.00     20568
           4       1.00      1.00      1.00     10083
           5       1.00      1.00      1.00     23066
           6       1.00      1.00      1.00     13023
           7       1.00      1.00      1.00     11637
           8       1.00      1.00      1.00     29453
           9       1.00    

### Affichage de la matrice de confusion

In [49]:
np.round( confusion_matrix(y3_test, predictions_HrsOut), 3)

array([[ 6519,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0, 65575,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0, 41745,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     2, 43794,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     0,     0, 13201,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     0,     0,     0, 36938,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     0,     0,     0,     0, 16

# test optimisation, peut-on se contenter de forêt aléatoire pour le calcul (tentative réduction temps)

In [50]:
cls_fda_Hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=220, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest"]},
    memory_limit=24576)

cls_fda_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=220, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest"]},
    memory_limit=24576)

cls_fda_HrsOut = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=220, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest"]},
    memory_limit=24576)



## Affinage des modèles

### prédiction Chasse

In [51]:
cls_fda_Hunt.fit(x_train, y_train, dataset_name='Cat Data')

predictions_fda_Hunt = list(cls_fda_Hunt.predict(x_test))

cls_fda_Hunt.leaderboard()

[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 367 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 426 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 88 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 605 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 37 not found
[WARNING] [2025-04-27 07:37:57,280:Client-AutoMLSMBO(1)::Cat Data] Configuration 310 not found
[WARNING] [2025-04-27 07:37:57,281:Client-AutoMLSMBO(1)::Cat Data] Configuration 437 not found
[WARNING] [2025-04-27 07:37:57,281:Client-AutoMLSMBO(1)::Cat Data] Configuration 142 not found
[WARNING] [2025-04-27 07:37:57,281:Client-AutoMLSMBO

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
1,1,1.0,<NA>,<NA>,<NA>


résultats : peut pas faire d'arbre avec ces données

### prédiction N.pray

In [52]:
cls_fda_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

predictions_fda_Npray = list(cls_fda_Npray.predict(x2_test))

cls_fda_Npray.leaderboard()

[WARNING] [2025-04-27 07:41:34,039:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 07:41:34,039:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 07:41:34,039:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 07:41:34,039:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 07:41:34,039:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 07:41:34,040:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not found
[WARNING] [2025-04-27 07:41:34,040:Client-AutoMLSMBO(1)::Cat Data] Configuration 434 not found
[WARNING] [2025-04-27 07:41:34,040:Client-AutoMLSMBO(1)::Cat Data] Configuration 17 not found
[WARNING] [2025-04-27 07:41:34,040:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 07:41:34,040:Client-AutoMLSMBO(1)::Cat Data] Configuration 277 not found
[WARNING] [2025-04-27 07:41:34,040:Client-AutoMLSMBO

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
1,1,1.0,<NA>,<NA>,<NA>


résultats : peut pas faire d'arbre avec ces données

### prédiction Hrs.indors

In [53]:
cls_fda_HrsOut.fit(x3_train, y3_train, dataset_name='Cat Data')

predictions_fda_HrsOut = list(cls_fda_HrsOut.predict(x3_test))

cls_fda_HrsOut.leaderboard()

[WARNING] [2025-04-27 07:45:10,213:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 07:45:10,213:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 07:45:10,213:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 07:45:10,213:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 07:45:10,213:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 07:45:10,214:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not found
[WARNING] [2025-04-27 07:45:10,214:Client-AutoMLSMBO(1)::Cat Data] Configuration 17 not found
[WARNING] [2025-04-27 07:45:10,214:Client-AutoMLSMBO(1)::Cat Data] Configuration 434 not found
[WARNING] [2025-04-27 07:45:10,214:Client-AutoMLSMBO(1)::Cat Data] Configuration 615 not found
[WARNING] [2025-04-27 07:45:10,214:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 07:45:10,214:Client-AutoMLSMBO

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
1,1,1.0,<NA>,<NA>,<NA>


résultats : peut pas faire d'arbre avec ces données

## affichage des résultats

### Hunt

In [54]:
print( sklearn.metrics.classification_report(y_test, predictions_fda_Hunt) )

np.round( confusion_matrix(y_test, predictions_fda_HrsOut), 3)

/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

          Na       0.14      1.00      0.24     72285
          No       0.00      0.00      0.00     75535
         Yes       0.00      0.00      0.00    377823

    accuracy                           0.14    525643
   macro avg       0.05      0.33      0.08    525643
weighted avg       0.02      0.14      0.03    525643



array([[     0,      0,      0,      0],
       [ 72285,      0,      0,      0],
       [ 75535,      0,      0,      0],
       [377823,      0,      0,      0]])

### N.pray

In [55]:
print( sklearn.metrics.classification_report(y2_test, predictions_fda_Npray) )

np.round( confusion_matrix(y2_test, predictions_fda_Npray), 3)

/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

          -1       0.31      1.00      0.47    163262
           0       0.00      0.00      0.00    106831
           1       0.00      0.00      0.00     53779
          10       0.00      0.00      0.00     16599
          11       0.00      0.00      0.00      2699
          13       0.00      0.00      0.00      1214
          15       0.00      0.00      0.00      3493
          16       0.00      0.00      0.00       808
          17       0.00      0.00      0.00       561
          19       0.00      0.00      0.00      7307
           2       0.00      0.00      0.00     53685
           3       0.00      0.00      0.00     28708
           4       0.00      0.00      0.00     41843
           5       0.00      0.00      0.00     19360
           6       0.00      0.00      0.00     11099
           7       0.00      0.00      0.00      3248
           8       0.00      0.00      0.00      8978
           9       0.00    

array([[163262,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [106831,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [ 53779,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [ 16599,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [  2699,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [  1214,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
           

### Hrs.indors

In [56]:
print( sklearn.metrics.classification_report(y3_test, predictions_fda_HrsOut) )

np.round( confusion_matrix(y3_test, predictions_fda_HrsOut), 3)

/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

           0       0.01      1.00      0.02      6519
           1       0.00      0.00      0.00     65575
          10       0.00      0.00      0.00     41745
          12       0.00      0.00      0.00     43796
          14       0.00      0.00      0.00     13201
          15       0.00      0.00      0.00     36938
          16       0.00      0.00      0.00     16434
          18       0.00      0.00      0.00     53462
           2       0.00      0.00      0.00     52736
          20       0.00      0.00      0.00     12525
          23       0.00      0.00      0.00      7306
           3       0.00      0.00      0.00     20568
           4       0.00      0.00      0.00     10083
           5       0.00      0.00      0.00     23066
           6       0.00      0.00      0.00     13023
           7       0.00      0.00      0.00     11637
           8       0.00      0.00      0.00     29453
           9       0.00    

array([[ 6519,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [65575,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [41745,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [43796,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [13201,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [36938,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [16434,     0,     0,     0,     0,     0,   

## Conclu : 



Impossibilité de faire un arbre pour prédire quelque soit la 3 prédiction

# test optimisation 2, peut-on se contenter de k plus proche voisins pour le calcul (tentative réduction temps)

In [32]:
cls_kN_Hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["k_nearest_neighbors"]},
    memory_limit=24576)

cls_kN_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["k_nearest_neighbors"]},
    memory_limit=24576)

cls_kN_HrsOut = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["k_nearest_neighbors"]},
    memory_limit=24576)

## Affinage des modèles

### prédiction Chasse

In [33]:
cls_kN_Hunt.fit(x_train, y_train, dataset_name='Cat Data')

predictions_kN_Hunt = list(cls_kN_Hunt.predict(x_test))

[WARNING] [2025-04-27 08:33:08,644:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 367 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 426 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 88 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 605 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 266 not found
[WARNING] [2025-04-27 08:33:10,156:Client-AutoMLSMBO(1)::Cat Data] Configuration 37 not f

In [59]:
cls_kN_Hunt.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
3,1,0.16,k_nearest_neighbors,0.000000,34.553523
7,2,0.12,k_nearest_neighbors,0.000000,24.677164
9,3,0.18,k_nearest_neighbors,0.000000,30.720862
12,4,0.06,k_nearest_neighbors,0.000000,38.710556
25,5,0.06,k_nearest_neighbors,0.000000,70.377589
27,6,0.08,k_nearest_neighbors,0.000000,69.085729
31,7,0.06,k_nearest_neighbors,0.000000,31.255920
17,8,0.10,k_nearest_neighbors,0.000006,39.188300
29,9,0.06,k_nearest_neighbors,0.000144,42.360525


### prédiction N.pray

In [60]:
cls_kN_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

predictions_kN_Npray = list(cls_kN_Npray.predict(x2_test))

[WARNING] [2025-04-27 07:53:58,970:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 07:54:00,442:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 07:54:00,442:Client-AutoMLSMBO(1)::Cat Data] Configuration 509 not found
[WARNING] [2025-04-27 07:54:00,442:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 07:54:00,443:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 07:54:00,444:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 07:54:00,444:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 07:54:00,444:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 07:54:00,444:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not found
[WARNING] [2025-04-27 07:54:00,444:Client-AutoMLSMBO(1)::Cat Data] Configuration 434 not 

In [61]:
cls_kN_Npray.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
3,1,0.16,k_nearest_neighbors,0.000000,29.903038
4,2,0.12,k_nearest_neighbors,0.000000,19.784661
10,3,0.18,k_nearest_neighbors,0.000000,33.792944
20,4,0.06,k_nearest_neighbors,0.000000,24.441357
27,5,0.06,k_nearest_neighbors,0.000000,13.280283
17,6,0.10,k_nearest_neighbors,0.000006,41.164173
22,7,0.08,k_nearest_neighbors,0.000006,38.883449
12,8,0.06,k_nearest_neighbors,0.000012,41.567254
23,9,0.06,k_nearest_neighbors,0.000023,10.584699


### prédiction Hrs.indors

In [62]:
cls_kN_HrsOut.fit(x3_train, y3_train, dataset_name='Cat Data')

predictions_kN_HrsOut = list(cls_kN_HrsOut.predict(x3_test))

[WARNING] [2025-04-27 07:58:16,314:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 07:58:17,826:Client-AutoMLSMBO(1)::Cat Data] Configuration 509 not found
[WARNING] [2025-04-27 07:58:17,826:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not found
[WARNING] [2025-04-27 07:58:17,827:Client-AutoMLSMBO(1)::Cat Data] Configuration 17 not f

In [63]:
cls_kN_HrsOut.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
5,1,0.16,k_nearest_neighbors,0.000000,21.723667
20,2,0.06,k_nearest_neighbors,0.000000,30.577849
3,3,0.14,k_nearest_neighbors,0.000006,29.874283
10,4,0.16,k_nearest_neighbors,0.000006,29.773019
23,5,0.10,k_nearest_neighbors,0.000006,13.770110
30,6,0.06,k_nearest_neighbors,0.000006,29.848074
17,7,0.08,k_nearest_neighbors,0.000012,44.006118
25,8,0.06,k_nearest_neighbors,0.000017,17.193578
12,9,0.06,k_nearest_neighbors,0.000023,40.137983


## affichage des résultats

### Hunt

In [34]:
print( sklearn.metrics.classification_report(y_test, predictions_kN_Hunt) )

np.round( confusion_matrix(y_test, predictions_kN_Hunt), 3)

              precision    recall  f1-score   support

          Na       1.00      1.00      1.00     72285
          No       1.00      1.00      1.00     75535
         Yes       1.00      1.00      1.00    377823

    accuracy                           1.00    525643
   macro avg       1.00      1.00      1.00    525643
weighted avg       1.00      1.00      1.00    525643



array([[ 72285,      0,      0],
       [     0,  75535,      0],
       [     0,      0, 377823]])

### N.pray

In [65]:
print( sklearn.metrics.classification_report(y2_test, predictions_kN_Npray) )

np.round( confusion_matrix(y2_test, predictions_kN_Npray), 3)

              precision    recall  f1-score   support

          -1       1.00      1.00      1.00    163262
           0       1.00      1.00      1.00    106831
           1       1.00      1.00      1.00     53779
          10       1.00      1.00      1.00     16599
          11       1.00      1.00      1.00      2699
          13       1.00      1.00      1.00      1214
          15       1.00      1.00      1.00      3493
          16       1.00      1.00      1.00       808
          17       1.00      1.00      1.00       561
          19       1.00      1.00      1.00      7307
           2       1.00      1.00      1.00     53685
           3       1.00      1.00      1.00     28708
           4       1.00      1.00      1.00     41843
           5       1.00      1.00      1.00     19360
           6       1.00      1.00      1.00     11099
           7       1.00      1.00      1.00      3248
           8       1.00      1.00      1.00      8978
           9       1.00    

array([[163262,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0, 106831,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,  53779,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,  16599,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,      0,   2699,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,      0,      0,   1214,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
           

### Hrs.indors

In [66]:
print( sklearn.metrics.classification_report(y3_test, predictions_kN_HrsOut) )

np.round( confusion_matrix(y3_test, predictions_kN_HrsOut), 3)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6519
           1       1.00      1.00      1.00     65575
          10       1.00      1.00      1.00     41745
          12       1.00      1.00      1.00     43796
          14       1.00      1.00      1.00     13201
          15       1.00      1.00      1.00     36938
          16       1.00      1.00      1.00     16434
          18       1.00      1.00      1.00     53462
           2       1.00      1.00      1.00     52736
          20       1.00      1.00      1.00     12525
          23       1.00      1.00      1.00      7306
           3       1.00      1.00      1.00     20568
           4       1.00      1.00      1.00     10083
           5       1.00      1.00      1.00     23066
           6       1.00      1.00      1.00     13023
           7       1.00      1.00      1.00     11637
           8       1.00      1.00      1.00     29453
           9       1.00    

array([[ 6519,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0, 65575,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0, 41745,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     2, 43794,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     0,     0, 13201,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     0,     0,     0, 36938,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,     0,     0,     0,     0,     0, 16

## Conclusion : 

Utiliser les k plus proches voisins donne des résultats bien plus rapidement tout aussi juste

	==> pas nécécaire de perdre de la puissance de calcul pour les autres méthodes

# test optimisation 3, peut-on se contenter du naïf Baésien pour le calcul (tentative réduction temps)

In [67]:
cls_nb_Hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["gaussian_nb"]},
    memory_limit=24576)

cls_nb_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["gaussian_nb"]},
    memory_limit=24576)

cls_nb_HrsOut = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["gaussian_nb"]},
    memory_limit=24576)

## Affinage des modèles

### prédiction Chasse

In [68]:
cls_nb_Hunt.fit(x_train, y_train, dataset_name='Cat Data')

predictions_nb_Hunt = list(cls_nb_Hunt.predict(x_test))

[WARNING] [2025-04-27 08:02:59,148:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:03:00,659:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:03:00,660:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-27 08:03:00,660:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:03:00,661:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 08:03:00,661:Client-AutoMLSMBO(1)::Cat Data] Configuration 367 not found
[WARNING] [2025-04-27 08:03:00,661:Client-AutoMLSMBO(1)::Cat Data] Configuration 426 not found
[WARNING] [2025-04-27 08:03:00,662:Client-AutoMLSMBO(1)::Cat Data] Configuration 88 not found
[WARNING] [2025-04-27 08:03:00,662:Client-AutoMLSMBO(1)::Cat Data] Configuration 605 not found
[WARNING] [2025-04-27 08:03:00,662:Client-AutoMLSMBO(1)::Cat Data] Configuration 266 not 

In [69]:
cls_nb_Hunt.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
42,1,0.04,gaussian_nb,0.141547,49.158720
27,2,0.12,gaussian_nb,0.147900,5.489612
11,3,0.02,gaussian_nb,0.149330,8.356408
23,4,0.02,gaussian_nb,0.149330,5.712638
66,5,0.10,gaussian_nb,0.149439,4.506705
39,6,0.02,gaussian_nb,0.156374,11.986636
5,7,0.04,gaussian_nb,0.156945,18.909016
2,11,0.04,gaussian_nb,0.162779,4.585874
4,13,0.02,gaussian_nb,0.162779,5.591912


### prédiction N.pray

In [70]:
cls_nb_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

predictions_nb_Npray = list(cls_nb_Npray.predict(x2_test))

[WARNING] [2025-04-27 08:06:34,336:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:06:35,998:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:06:35,998:Client-AutoMLSMBO(1)::Cat Data] Configuration 509 not found
[WARNING] [2025-04-27 08:06:35,998:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:06:35,999:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 08:06:35,999:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 08:06:35,999:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 08:06:35,999:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 08:06:35,999:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 08:06:35,999:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not 

In [71]:
cls_nb_Npray.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
45,1,0.22,gaussian_nb,0.487392,26.777017
47,2,0.02,gaussian_nb,0.514043,23.356947
30,3,0.02,gaussian_nb,0.518033,5.238057
24,4,0.08,gaussian_nb,0.573970,32.354060
46,5,0.16,gaussian_nb,0.576288,30.306376
4,6,0.02,gaussian_nb,0.593571,37.151892
26,7,0.46,gaussian_nb,0.620389,3.282469
18,8,0.02,gaussian_nb,0.625001,12.675972


### prédiction Hrs.indors

In [72]:
cls_nb_HrsOut.fit(x3_train, y3_train, dataset_name='Cat Data')

predictions_nb_HrsOut = list(cls_nb_HrsOut.predict(x3_test))

[WARNING] [2025-04-27 08:09:59,685:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:10:01,299:Client-AutoMLSMBO(1)::Cat Data] Configuration 509 not found
[WARNING] [2025-04-27 08:10:01,299:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:10:01,300:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 08:10:01,300:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 08:10:01,300:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:10:01,300:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 08:10:01,301:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 08:10:01,301:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 08:10:01,301:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not 

In [73]:
cls_nb_HrsOut.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
20,1,0.32,gaussian_nb,0.664532,13.342049
18,2,0.08,gaussian_nb,0.758304,10.848340
15,3,0.24,gaussian_nb,0.765049,9.024784
7,4,0.08,gaussian_nb,0.803744,6.021194
8,5,0.22,gaussian_nb,0.814616,6.521437
9,6,0.06,gaussian_nb,0.864501,16.356224


## affichage des résultats

### Hunt

In [74]:
print( sklearn.metrics.classification_report(y_test, predictions_nb_Hunt) )

np.round( confusion_matrix(y_test, predictions_nb_Hunt), 3)

              precision    recall  f1-score   support

          Na       0.71      0.66      0.68     72285
          No       0.73      0.72      0.72     75535
         Yes       0.93      0.95      0.94    377823

    accuracy                           0.88    525643
   macro avg       0.79      0.78      0.78    525643
weighted avg       0.87      0.88      0.88    525643



array([[ 47515,  10437,  14333],
       [ 10456,  54240,  10839],
       [  9243,   9515, 359065]])

### N.pray

In [75]:
print( sklearn.metrics.classification_report(y2_test, predictions_nb_Npray) )

np.round( confusion_matrix(y2_test, predictions_nb_Npray), 3)

              precision    recall  f1-score   support

          -1       0.88      0.81      0.85    163262
           0       0.43      0.61      0.51    106831
           1       0.53      0.23      0.33     53779
          10       0.45      0.53      0.49     16599
          11       0.20      0.61      0.30      2699
          13       1.00      1.00      1.00      1214
          15       1.00      0.83      0.91      3493
          16       1.00      0.99      0.99       808
          17       1.00      1.00      1.00       561
          19       0.79      0.55      0.65      7307
           2       0.43      0.49      0.45     53685
           3       0.44      0.17      0.24     28708
           4       0.47      0.44      0.45     41843
           5       0.32      0.35      0.34     19360
           6       0.42      0.35      0.38     11099
           7       0.22      0.76      0.34      3248
           8       0.68      0.61      0.64      8978
           9       1.00    

array([[132400,  14837,      0,    672,      0,      0,      0,      0,
             0,      0,   6870,   1308,   6018,    505,      0,      0,
           652,      0],
       [ 14415,  65108,   2335,   3887,   2262,      0,      0,      0,
             0,    462,   7158,      0,   3170,   6895,      0,   1139,
             0,      0],
       [  2670,  18905,  12602,   2813,   1281,      0,      0,      0,
             0,      0,   5263,   1018,   2926,   1831,      0,   3273,
          1197,      0],
       [     0,   3222,      1,   8855,      0,      0,      0,      0,
             0,      0,   3648,    139,      0,      3,      0,      0,
           731,      0],
       [     0,   1065,      0,      0,   1634,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      2,      0,      0,   1212,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
           

### Hrs.indors

In [76]:
print( sklearn.metrics.classification_report(y3_test, predictions_nb_HrsOut) )

np.round( confusion_matrix(y3_test, predictions_nb_HrsOut), 3)

              precision    recall  f1-score   support

           0       0.57      0.67      0.62      6519
           1       0.53      0.34      0.41     65575
          10       0.19      0.20      0.19     41745
          12       0.60      0.32      0.42     43796
          14       0.22      0.40      0.28     13201
          15       0.22      0.67      0.33     36938
          16       0.44      0.25      0.32     16434
          18       0.35      0.07      0.11     53462
           2       0.57      0.35      0.43     52736
          20       0.20      0.87      0.33     12525
          23       0.22      0.91      0.36      7306
           3       0.29      0.24      0.27     20568
           4       0.20      0.72      0.31     10083
           5       0.62      0.21      0.31     23066
           6       0.31      0.49      0.38     13023
           7       0.36      0.67      0.47     11637
           8       0.82      0.68      0.75     29453
           9       0.20    

array([[ 4385,   860,     0,     0,     0,     0,     0,     0,     0,
            0,   498,   636,     0,     0,     0,   140,     0,     0,
            0],
       [    0, 22096,  3121,  3709,  2614, 15522,  1711,     0,  2609,
         8958,     0,  2053,  1644,     0,  1010,   528,     0,     0,
            0],
       [    0,  1958,  8160,     0,  1267, 10107,     0,     0,   831,
         4191,  4007,  3349,  2763,     0,  1093,  2019,     0,  2000,
            0],
       [    0,   148,  1424, 14140,  3003,  7768,  1517,  1437,   801,
         3840,  2163,   418,  2651,     0,   619,  2208,     0,  1103,
          556],
       [    0,   672,     0,     0,  5243,  2854,     0,  1552,     0,
            0,  2880,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,  1003,   672,  1553,  1092, 24895,   607,     0,     0,
         3567,     0,  1468,  1669,     1,     1,   410,     0,     0,
            0],
       [  824,   397,   744,   633,     0,  4863,  4

## Conclusion : 

Utiliser le naif bayésien donne des résultats bien plus rapidement mais bien plux faux

	==> pas nécécaire de perdre de la puissance de calcul pour les autres méthodes

# test optimisation 4, peut-on se contenter du lda (linear discrinant analysis) pour le calcul (tentative réduction temps)

In [20]:
cls_lda_Hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["lda"]},
    memory_limit=24576)

cls_lda_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["lda"]},
    memory_limit=24576)

cls_lda_HrsOut = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=200, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["lda"]},
    memory_limit=24576)

## Affinage des modèles

### prédiction Chasse

In [21]:
cls_lda_Hunt.fit(x_train, y_train, dataset_name='Cat Data')

predictions_lda_Hunt = list(cls_lda_Hunt.predict(x_test))

[WARNING] [2025-04-27 08:19:15,389:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 367 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 426 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 88 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 605 not found
[WARNING] [2025-04-27 08:19:16,823:Client-AutoMLSMBO(1)::Cat Data] Configuration 266 not 

In [22]:
cls_lda_Hunt.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
14,1,0.64,lda,0.065155,30.068367
9,2,0.02,lda,0.141420,14.633729
6,3,0.12,lda,0.155677,9.350123
11,4,0.02,lda,0.213926,9.104187
19,5,0.02,lda,0.213926,7.669858
23,6,0.02,lda,0.213926,4.523256
12,7,0.02,lda,0.220671,2.435512
18,8,0.04,lda,0.222452,4.279077
20,9,0.02,lda,0.222452,4.557085


### prédiction N.pray

In [23]:
cls_lda_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

predictions_lda_Npray = list(cls_lda_Npray.predict(x2_test))

[WARNING] [2025-04-27 08:22:51,728:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 509 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-27 08:22:53,456:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not 

In [24]:
cls_lda_Npray.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
14,1,0.24,lda,0.262052,46.252483
31,2,0.22,lda,0.321194,20.014729
21,3,0.16,lda,0.516741,15.853187
20,4,0.02,lda,0.553925,27.678117
27,5,0.30,lda,0.577320,9.872222
12,6,0.06,lda,0.585298,7.051345


### prédiction Hrs.indors

In [25]:
cls_lda_HrsOut.fit(x3_train, y3_train, dataset_name='Cat Data')

predictions_lda_HrsOut = list(cls_lda_HrsOut.predict(x3_test))

[WARNING] [2025-04-27 08:26:22,432:Client-AutoML(1):Cat Data] Capping the per_run_time_limit to 99.0 to have time for a least 2 models in each process.
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 509 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 322 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 543 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 19 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 553 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 628 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 288 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 298 not found
[WARNING] [2025-04-27 08:26:23,950:Client-AutoMLSMBO(1)::Cat Data] Configuration 647 not 

In [26]:
cls_lda_HrsOut.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
14,1,0.20,lda,0.337509,38.072702
24,2,0.14,lda,0.468402,10.479599
23,3,0.14,lda,0.600719,19.015767
20,4,0.02,lda,0.719812,8.733480
2,5,0.02,lda,0.721507,5.211615
19,6,0.02,lda,0.721507,19.498872
18,7,0.04,lda,0.732466,5.764215
30,8,0.12,lda,0.736628,5.778673
12,9,0.04,lda,0.763752,4.961150


## affichage des résultats

### Hunt

In [27]:
print( sklearn.metrics.classification_report(y_test, predictions_lda_Hunt) )

np.round( confusion_matrix(y_test, predictions_lda_Hunt), 3)

              precision    recall  f1-score   support

          Na       0.87      0.81      0.84     72285
          No       0.83      0.90      0.86     75535
         Yes       0.97      0.96      0.96    377823

    accuracy                           0.93    525643
   macro avg       0.89      0.89      0.89    525643
weighted avg       0.93      0.93      0.93    525643



array([[ 58856,   5646,   7783],
       [  3137,  68087,   4311],
       [  5923,   8558, 363342]])

### N.pray

In [28]:
print( sklearn.metrics.classification_report(y2_test, predictions_lda_Npray) )

np.round( confusion_matrix(y2_test, predictions_lda_Npray), 3)

              precision    recall  f1-score   support

          -1       0.91      0.89      0.90    163262
           0       0.67      0.70      0.68    106831
           1       0.72      0.66      0.69     53779
          10       0.71      0.83      0.76     16599
          11       0.51      0.61      0.56      2699
          13       0.24      1.00      0.39      1214
          15       0.55      1.00      0.71      3493
          16       0.47      1.00      0.64       808
          17       1.00      1.00      1.00       561
          19       0.82      0.68      0.74      7307
           2       0.60      0.74      0.66     53685
           3       0.80      0.50      0.62     28708
           4       0.78      0.79      0.78     41843
           5       0.67      0.56      0.61     19360
           6       1.00      0.59      0.74     11099
           7       0.87      0.76      0.81      3248
           8       0.69      0.74      0.72      8978
           9       1.00    

array([[144615,   6067,   1494,    672,      0,      0,   1204,      0,
             0,      0,   6154,      0,   1181,      0,      0,      0,
          1875,      0],
       [ 13064,  74969,   1026,   1331,   1544,    587,   1164,      0,
             0,    462,   7064,   1848,   1072,   1597,      0,      1,
          1102,      0],
       [  1153,   6498,  35661,    743,      0,   2508,      0,      0,
             0,      0,   4814,    450,   1593,      0,      0,    359,
             0,      0],
       [     0,   1828,      0,  13750,      0,      0,    472,      0,
             0,      0,    549,      0,      0,      0,      0,      0,
             0,      0],
       [     0,    340,      0,      0,   1634,      0,      0,      0,
             0,      0,    725,      0,      0,      0,      0,      0,
             0,      0],
       [     0,      0,      0,      0,      0,   1214,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
           

### Hrs.indors

In [29]:
print( sklearn.metrics.classification_report(y3_test, predictions_lda_HrsOut) )

np.round( confusion_matrix(y3_test, predictions_lda_HrsOut), 3)

              precision    recall  f1-score   support

           0       0.86      0.77      0.81      6519
           1       0.50      0.78      0.61     65575
          10       0.68      0.52      0.59     41745
          12       0.71      0.68      0.69     43796
          14       0.86      0.83      0.84     13201
          15       0.58      0.75      0.66     36938
          16       0.72      0.53      0.61     16434
          18       0.74      0.70      0.72     53462
           2       0.70      0.60      0.65     52736
          20       0.76      0.61      0.68     12525
          23       0.41      0.70      0.51      7306
           3       0.91      0.76      0.83     20568
           4       0.62      0.92      0.74     10083
           5       0.96      0.63      0.76     23066
           6       0.99      0.73      0.84     13023
           7       0.82      0.72      0.76     11637
           8       1.00      0.81      0.90     29453
           9       0.63    

array([[ 5021,   860,     0,     0,     0,     0,     0,     0,   140,
            0,   498,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0, 51065,  1182,  1403,     0,  4164,     0,  1744,   831,
            0,  1706,     0,     0,     1,     0,     0,     0,  3479,
            0],
       [    0,  6697, 21752,   951,  1086,  1963,     0,  3288,  2320,
            0,     0,     0,   181,     0,     0,   534,     0,  2763,
          210],
       [    0,  2001,   155, 29862,     0,  2004,  1976,     0,   603,
            1,   342,     0,   861,   418,     0,    32,     0,  5541,
            0],
       [    0,   672,     0,     0, 10964,    27,     0,  1538,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    0,  2030,   921,   734,     0, 27796,     0,     0,     0,
            0,   834,  1468,  1669,     0,     0,     0,     0,  1486,
            0],
       [  824,  3402,   177,     0,     0,     0,  8

## Conclusion : 